# Mandala.AI - Colab (NVIDIA T4) Setup

Notebook ini menyiapkan environment untuk menjalankan `main-api.py` dan `backend/` pada Google Colab dengan GPU NVIDIA T4. Ikuti langkah di bawah ini: clone repo atau upload file, install dependencies, jalankan server, dan expose dengan ngrok.

Catatan penting: Colab runtime bersifat ephemeral — gunakan untuk eksperimen/development saja.

In [ ]:
# 1) Clone repository atau upload zip. Ubah <your-repo-url> jika perlu
!git clone https://github.com/your-username/your-repo.git mandala_repo || true
%cd mandala_repo || true

In [ ]:
# 2) Set runtime to GPU (do this via Runtime->Change runtime type)
import os
print('CUDA_VISIBLE_DEVICES=', os.environ.get('CUDA_VISIBLE_DEVICES', None))

In [ ]:
# 3) Install / upgrade pip and basic deps. We pin torch for T4 (CUDA 11.8 / cu118)
!python -m pip install --upgrade pip setuptools wheel
!pip install -r requirements.txt || true
# Install torch for CUDA 11.8 (compatible with T4). If already present, this will skip or upgrade.
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118 -U
!pip install nvidia-ml-py3 pyngrok==5.1.0 --upgrade

In [ ]:
# 4) Quick GPU & NVML check
import torch
from pynvml import nvmlInit, nvmlDeviceGetCount, nvmlDeviceGetName
print('torch version:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0))
nvmlInit()
print('nvml device count:', nvmlDeviceGetCount())

In [ ]:
# 5) Run the API server (uvicorn) in background and capture logs
# Ensure you're in the directory containing main-api.py
import os, subprocess, time
logfile = 'uvicorn_colab.log'
cmd = ['nohup','python','-u','-c',
       "import uvicorn; uvicorn.run('main-api:app', host='0.0.0.0', port=8000, log_level='info')"]
# Use nohup-like behavior by launching via subprocess and redirecting output to file
with open(logfile, 'w') as f:
    proc = subprocess.Popen(['uvicorn','main-api:app','--host','0.0.0.0','--port','8000','--log-level','info'], stdout=f, stderr=subprocess.STDOUT)
time.sleep(2)
print('Started uvicorn (pid):', proc.pid)
print('Tailing log file (first 20 lines):')
!sleep 1 || true
!head -n 20 uvicorn_colab.log || true

In [ ]:
# 6) Expose port via ngrok and print public URL.
from pyngrok import ngrok
# Optional: set your auth token for persistent tunnels
ngrok.set_auth_token('3JoKaviPKruVA4DnAVFAaskK3RL_6erEevy4Zx1w3rsUdnFCd')
tunnel = ngrok.connect(8000, bind_tls=True)
print('Public URL:', tunnel.public_url)

In [ ]:
# 7) Sanity-check health endpoint via the public URL
import requests, json
public = tunnel.public_url
try:
    r = requests.get(public + '/api/v1/health', timeout=10)
    print('health:', r.status_code, r.text)
except Exception as e:
    print('health check failed:', e)

## Cleanup
Jika ingin menghentikan server dan tunnel: hentikan proses UVicorn (lihat PID) dan panggil `ngrok.disconnect(public_url)` dan `ngrok.kill()`.